# C5 Grid Search — UA-ASL Hyperparameter Tuning
**γ⁺ = 0 (fixed) × γ⁻ ∈ {2,4,6} × λu ∈ {0.3,0.5,0.7} × α ∈ {0.3,0.5} → 18 runs** (theo `grid_search_c5.py`)  
Metric chọn best: `unc_AUC` trên `val_uncertain`  
Mỗi run: số epoch lấy từ `cfg['grid_search']['epochs_per_run']` (mặc định 5) × train CSV trong config

In [ ]:
import os, sys

WORK  = '/kaggle/working'
REPO  = 'https://github.com/PhuongThao-2005/TULIP-MedML.git'
RNAME = 'TULIP-MedML'
BRANCH = 'grid-search'

if os.path.exists(f'{WORK}/{RNAME}'):
    os.system(f'cd {WORK}/{RNAME} && git fetch origin')
    os.system(f'cd {WORK}/{RNAME} && git checkout {BRANCH}')
    os.system(f'cd {WORK}/{RNAME} && git pull origin {BRANCH}')
    print(f'Pulled latest ({BRANCH})')
else:
    os.system(f'cd {WORK} && git clone -b {BRANCH} {REPO}')
    print(f'Cloned ({BRANCH})')

os.system('pip install pyyaml timm -q')
sys.path.insert(0, f'{WORK}/{RNAME}')
os.chdir(f'{WORK}/{RNAME}')

print('Working dir :', os.getcwd())
print('Repo contents:', os.listdir('.'))

In [ ]:
# ── 0. Paths — chỉ cần chỉnh block này ──────────────────────────────────────
import sys, os

PROJECT_ROOT = "/kaggle/working/TULIP-MedML"   # root của project
CONFIG_PATH  = f"{PROJECT_ROOT}/src/configs/c5_tulip.yaml"
DATA_ROOT    = "/kaggle/input/datasets/ashery/chexpert"  # optional override

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working dir:", os.getcwd())

In [ ]:
# ── 2. Run grid search (THỰC) ────────────────────────────────────────────────
# ~18 runs × epochs/run — kết quả + checkpoint vào cfg['output']['log_dir']
# n_epochs=None → dùng cfg['grid_search']['epochs_per_run']; dry_run=True chỉ test pipeline

from src.grid_search_c5 import run_grid_search

results, best = run_grid_search(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    n_epochs=5,
    # dry_run=True,
)

In [ ]:
# ── 3. Load kết quả đã lưu (nếu resume từ session khác) ─────────────────────
import json, yaml

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)
LOG_DIR = _cfg["output"]["log_dir"]

with open(f"{LOG_DIR}/grid_results.json") as f:
    results = json.load(f)

with open(f"{LOG_DIR}/best_params.json") as f:
    best = json.load(f)

print(f"Loaded {len(results)} runs.  Best unc_AUC = {best['unc_auc']:.4f}")
print(f"  γ+={best['gamma_pos']} (fixed)  γ-={best['gamma_neg']}  λu={best['lambda_unc']}  α={best['alpha']}")

In [ ]:
# ── 4a. Inline display — heatmaps (unc_AUC, mAP, mean_AUC) ──────────────────
from IPython.display import Image, display
import os

for metric in ["unc_auc", "mAP", "mean_auc"]:
    path = os.path.join(LOG_DIR, f"grid_heatmap_{metric}.png")
    if os.path.isfile(path):
        print(f"\n{'═'*50}")
        print(f"  Heatmap: {metric}")
        print(f"{'═'*50}")
        display(Image(filename=path, width=900))
    else:
        print(f"[missing] {path}")

In [ ]:
# ── 4b. Inline display — scatter + parallel coordinates ─────────────────────
from IPython.display import Image, display

for fname in ["grid_scatter.png", "grid_parallel.png"]:
    path = os.path.join(LOG_DIR, fname)
    if os.path.isfile(path):
        display(Image(filename=path, width=900))
    else:
        print(f"[missing] {path}")

In [ ]:
# ── 5. Pandas summary table — tất cả runs trong grid ───────────────────────
import pandas as pd

df = pd.DataFrame(results)[
    ["run_id", "gamma_pos", "gamma_neg", "lambda_unc", "alpha",
     "mAP", "mean_auc", "unc_auc", "elapsed_min"]
].sort_values("unc_auc", ascending=False).reset_index(drop=True)

# Highlight best row
def _highlight_best(row):
    is_best = (
        row["gamma_pos"]  == best["gamma_pos"] and
        row["gamma_neg"]  == best["gamma_neg"] and
        row["lambda_unc"] == best["lambda_unc"] and
        row["alpha"] == best["alpha"]
    )
    return ["background-color: #d4edda" if is_best else "" for _ in row]

df.style\
    .apply(_highlight_best, axis=1)\
    .format({"mAP": "{:.4f}", "mean_auc": "{:.4f}",
             "unc_auc": "{:.4f}", "elapsed_min": "{:.1f}"})\
    .background_gradient(subset=["unc_auc"], cmap="Blues")\
    .background_gradient(subset=["mAP"], cmap="Greens")\
    .background_gradient(subset=["mean_auc"], cmap="Oranges")\
    .set_caption("Grid search results — sorted by unc_AUC (best highlighted)")

In [ ]:
# ── 6. Re-generate plots bất cứ lúc nào (sau khi đã có results) ─────────────
from src.grid_search_c5 import _make_visualisations

_make_visualisations(results, LOG_DIR)
print("Plots regenerated →", LOG_DIR)

In [ ]:
# ── 7. In best params — paste vào config để train full ──────────────────────
print("Best hyperparameters (paste into c5_tulip.yaml):")
print()
print("loss:")
print(f"  type:       ua_asl")
print(f"  gamma_pos:  {best['gamma_pos']}")
print(f"  gamma_neg:  {best['gamma_neg']}")
print(f"  lambda_unc: {best['lambda_unc']}")
print(f"  alpha:      {best['alpha']}")
print()
print(f"Best unc_AUC  = {best['unc_auc']:.4f}")
print(f"Best mAP      = {best['mAP']:.4f}")
print(f"Best mean_AUC = {best['mean_auc']:.4f}")